# Week 5: Volatility Forecasting Model Variants 
*Objective:* Build upon baseline volatility forecasting model by incorporating network features (Model 2) and cluster-level sentiment features (Model 3) to predict 20-day forward realized volatility.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Setup paths
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

# 2. Load Datasets
# Market Returns (for baseline volatility)
market_df = pd.read_parquet(root / "data" / "processed" / "returns_market.parquet")
market_df.index = pd.to_datetime(market_df.index)

# Network/Cluster Features (from Grace)
cluster_df = pd.read_parquet(root / "data" / "processed" / "features" / "clusters_market_adjusted_20d.parquet")
cluster_df.index = pd.to_datetime(cluster_df.index)

# Sentiment Features (from Brandon's Week 4 deliverable)
sentiment_df = pd.read_parquet(root / "data" / "processed" / "features" / "cluster_level_sentiment.parquet")
# Group sentiment to a daily market level (mean of all clusters for the day)
daily_sentiment = sentiment_df.groupby('date')[['mean_sentiment']].mean()
daily_sentiment.index = pd.to_datetime(daily_sentiment.index)

print("Data loaded successfully.")

Data loaded successfully.


## 1. Feature Engineering & Target Definition
Following target definition, we calculate the 20-day forward realized volatility as our target variable. We also calculate the 20-day trailing volatility as our baseline feature, and align all datasets.

In [2]:
ANNUALIZATION = 252
H = 20

# Calculate Trailing (Baseline) and Forward (Target) Volatility
# Volatility = standard deviation of returns * sqrt(252)
volatility = market_df['SPY_ret'].rolling(window=H).std() * np.sqrt(ANNUALIZATION)

# Create the modeling dataset
model_data = pd.DataFrame(index=market_df.index)
model_data['trailing_vol_20d'] = volatility
model_data['target_vol_20d'] = volatility.shift(-H) # Shift backwards to predict the future

# Merge Network/Cluster features
model_data = model_data.join(cluster_df[['density', 'modularity', 'n_communities']])

# Merge Sentiment features
model_data = model_data.join(daily_sentiment)

# Drop NaNs created by rolling windows and shifting
model_data = model_data.dropna()

print(f"Modeling dataset ready. Shape: {model_data.shape}")
display(model_data.head())

Modeling dataset ready. Shape: (12, 6)


,trailing_vol_20d,target_vol_20d,density,modularity,n_communities,mean_sentiment
Date,,,,,,
2022-01-31,0.186754,0.223798,0.052030,0.665469,9.0,-0.137820
2022-03-01,0.223798,0.231187,0.050889,0.635851,8.0,-0.187034
2022-03-29,0.231187,0.195004,0.051109,0.627057,9.0,-0.216445
2022-04-27,0.195004,0.338374,0.051330,0.605699,7.0,-0.239458
2022-05-25,0.338374,0.322654,0.051098,0.578633,8.0,-0.347471


## 2. Model 2: Baseline + Network Features
We train a linear regression model using past volatility and network features (density, modularity, number of communities).

In [3]:
# Define features and target for Model 2
X2 = model_data[['trailing_vol_20d', 'density', 'modularity', 'n_communities']]
y = model_data['target_vol_20d']

# Train Model 2
model_2 = LinearRegression()
model_2.fit(X2, y)
preds_2 = model_2.predict(X2)

# Evaluate Model 2
mae_2 = mean_absolute_error(y, preds_2)
r2_2 = r2_score(y, preds_2)

print("--- Model 2: Baseline + Network Features ---")
print(f"MAE: {mae_2:.4f}")
print(f"R-squared: {r2_2:.4f}")

# Display feature coefficients
coef_df_2 = pd.DataFrame({'Feature': X2.columns, 'Coefficient': model_2.coef_})
display(coef_df_2)

--- Model 2: Baseline + Network Features ---
MAE: 0.0374
R-squared: 0.3886


,Feature,Coefficient
0,trailing_vol_20d,0.082288
1,density,78.200494
2,modularity,-0.986945
3,n_communities,-0.017167


## 3. Model 3: Baseline + Network + Sentiment
We train the final variant by adding the aggregated SEC sentiment scores to the Model 2 feature set.

In [4]:
# Define features and target for Model 3
X3 = model_data[['trailing_vol_20d', 'density', 'modularity', 'n_communities', 'mean_sentiment']]

# Train Model 3
model_3 = LinearRegression()
model_3.fit(X3, y)
preds_3 = model_3.predict(X3)

# Evaluate Model 3
mae_3 = mean_absolute_error(y, preds_3)
r2_3 = r2_score(y, preds_3)

print("--- Model 3: Baseline + Network + Sentiment ---")
print(f"MAE: {mae_3:.4f}")
print(f"R-squared: {r2_3:.4f}")

# Display feature coefficients
coef_df_3 = pd.DataFrame({'Feature': X3.columns, 'Coefficient': model_3.coef_})
display(coef_df_3)

--- Model 3: Baseline + Network + Sentiment ---
MAE: 0.0354
R-squared: 0.4984


,Feature,Coefficient
0,trailing_vol_20d,0.152997
1,density,61.438411
2,modularity,-1.202209
3,n_communities,-0.022334
4,mean_sentiment,0.339766
